# مسئلهٔ ۱ — V2 / مدل B: Fine-tuned ResNet18 + Mean–Max Pooling

A2 بهترین مدل frozen ما بود. در این مرحله classifier همان A2 را از checkpoint بارگذاری می‌کنیم و فقط آخرین block یعنی `layer4` در ResNet18 را با نرخ یادگیری بسیار کوچک باز می‌کنیم.

Augmentation فقط برای train و با **پارامتر مشترک برای تمام ۱۶ فریم یک sequence** اعمال می‌شود: crop ملایم، horizontal flip محدود و brightness/contrast ملایم. validation کاملاً deterministic است.

این مدل روی CPU سنگین است؛ batch size کوچک و gradient accumulation برای کنترل حافظه تنظیم شده‌اند.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2.csv'
MODEL_DIR = DATA_ROOT / 'models_v2'
MODEL_NAME = 'resnet18_meanmax_finetuned_layer4'
A2_CHECKPOINT_PATH = MODEL_DIR / 'resnet18_meanmax_pooling_frozen_best.pt'
A2_METRICS_PATH = MODEL_DIR / 'resnet18_meanmax_pooling_frozen_metrics.json'

NUM_FRAMES = 16
TARGET_HEIGHT = 224
TARGET_WIDTH = 320
FEATURE_DIM = 512
SEQUENCE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_EPOCHS = 5
PATIENCE = 2
BACKBONE_LR = 1e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
SEED = 42

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert SEQUENCE_MANIFEST_PATH.exists(), 'Run notebook 08 first.'
assert FRAME_INDEX_PATH.exists(), 'Run notebook 09 first.'
assert A2_CHECKPOINT_PATH.exists(), 'Run notebook 11 first.'
print(f'Device: {device}')

Device: cpu


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH).copy()
frame_index = pd.read_csv(FRAME_INDEX_PATH).copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
sequence_manifest['label'] = sequence_manifest['label'].astype(int)
frame_index['frame_valid'] = frame_index['frame_valid'].astype(str).str.lower().eq('true')
sequence_manifest = sequence_manifest.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)

assert len(sequence_manifest) == 600
assert len(frame_index) == 600 * NUM_FRAMES
assert frame_index['frame_valid'].all()
assert frame_index['frame_path'].map(lambda value: Path(value).is_file()).all()

frame_groups = {
    sequence_id: group.sort_values('frame_index').reset_index(drop=True)
    for sequence_id, group in frame_index.groupby('sequence_id', sort=False)
}
assert set(frame_groups) == set(sequence_manifest['sequence_id'])
assert all(len(group) == NUM_FRAMES for group in frame_groups.values())

train_table = sequence_manifest.loc[sequence_manifest['split'].eq('train')].reset_index(drop=True)
validation_table = sequence_manifest.loc[sequence_manifest['split'].eq('validation')].reset_index(drop=True)
assert len(train_table) == 480 and len(validation_table) == 120
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

label,0,1
split,,
train,240,240
validation,60,60


In [3]:
class CachedSequenceDataset(Dataset):
    def __init__(self, sequence_table: pd.DataFrame, grouped_frames: dict[str, pd.DataFrame], training: bool):
        self.sequence_table = sequence_table.reset_index(drop=True)
        self.grouped_frames = grouped_frames
        self.training = training

    def __len__(self) -> int:
        return len(self.sequence_table)

    @staticmethod
    def load_rgb(path: str) -> torch.Tensor:
        image_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f'Cannot read cached frame: {path}')
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(image_rgb.copy()).permute(2, 0, 1).float().div_(255.0)

    @staticmethod
    def sequence_consistent_augment(sequence: torch.Tensor) -> torch.Tensor:
        """One crop/flip/color choice is shared by every frame: [T, C, H, W]."""
        _, _, height, width = sequence.shape
        scale = random.uniform(0.90, 1.00)
        relative_ratio = random.uniform(0.95, 1.05)
        crop_area = height * width * scale
        crop_width = min(width, max(1, int(round((crop_area * (width / height) * relative_ratio) ** 0.5))))
        crop_height = min(height, max(1, int(round(crop_area / crop_width))))
        top = random.randint(0, height - crop_height)
        left = random.randint(0, width - crop_width)
        sequence = sequence[:, :, top:top + crop_height, left:left + crop_width]
        sequence = F.interpolate(sequence, size=(TARGET_HEIGHT, TARGET_WIDTH), mode='bilinear', align_corners=False)

        if random.random() < 0.30:
            sequence = torch.flip(sequence, dims=[3])
        brightness = random.uniform(0.90, 1.10)
        contrast = random.uniform(0.90, 1.10)
        sequence = (sequence * brightness).clamp(0.0, 1.0)
        global_mean = sequence.mean(dim=(0, 2, 3), keepdim=True)
        sequence = ((sequence - global_mean) * contrast + global_mean).clamp(0.0, 1.0)
        return sequence

    def __getitem__(self, index: int):
        row = self.sequence_table.iloc[index]
        frame_rows = self.grouped_frames[row.sequence_id]
        sequence = torch.stack([self.load_rgb(path) for path in frame_rows['frame_path']])
        assert sequence.shape == (NUM_FRAMES, 3, TARGET_HEIGHT, TARGET_WIDTH)
        if self.training:
            sequence = self.sequence_consistent_augment(sequence)
        sequence = (sequence - IMAGENET_MEAN) / IMAGENET_STD
        return sequence, int(row.label), row.sequence_id, row.video_id

train_dataset = CachedSequenceDataset(train_table, frame_groups, training=True)
validation_dataset = CachedSequenceDataset(validation_table, frame_groups, training=False)
images, label, sequence_id, video_id = train_dataset[0]
assert images.shape == (NUM_FRAMES, 3, TARGET_HEIGHT, TARGET_WIDTH)
print({'shape': tuple(images.shape), 'label': label, 'sequence_id': sequence_id, 'video_id': video_id})

{'shape': (16, 3, 224, 320), 'label': 1, 'sequence_id': 'V2-W2_0', 'video_id': '0'}


In [4]:
class FineTunedResNet18MeanMax(nn.Module):
    def __init__(self):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.LayerNorm(FEATURE_DIM * 2),
            nn.Dropout(0.35),
            nn.Linear(FEATURE_DIM * 2, 1),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, channels, height, width = images.shape
        frame_features = self.backbone(images.reshape(batch_size * time_steps, channels, height, width))
        frame_features = frame_features.reshape(batch_size, time_steps, FEATURE_DIM)
        mean_features = frame_features.mean(dim=1)
        max_features = frame_features.max(dim=1).values
        return self.classifier(torch.cat([mean_features, max_features], dim=1)).squeeze(1)

model = FineTunedResNet18MeanMax().to(device)
a2_checkpoint = torch.load(A2_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
a2_head_state = {key: value for key, value in a2_checkpoint['model_state_dict'].items() if key.startswith('classifier.')}
missing_keys, unexpected_keys = model.load_state_dict(a2_head_state, strict=False)
assert not unexpected_keys
assert all(key.startswith('backbone.') for key in missing_keys)

# Freeze everything, then unfreeze only layer4 and the trained A2 classifier.
for parameter in model.backbone.parameters():
    parameter.requires_grad_(False)
for parameter in model.backbone.layer4.parameters():
    parameter.requires_grad_(True)
for parameter in model.classifier.parameters():
    parameter.requires_grad_(True)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.layer4.parameters(), 'lr': BACKBONE_LR},
    {'params': model.classifier.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print({'trainable_parameters': trainable_parameters, 'loaded_a2_head': True})

{'trainable_parameters': 8396801, 'loaded_a2_head': True}


In [5]:
train_loader = DataLoader(train_dataset, batch_size=SEQUENCE_BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=SEQUENCE_BATCH_SIZE, shuffle=False, num_workers=0)

def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def evaluate(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, list[str]]:
    model.eval()
    labels_out, probabilities_out, sequence_ids_out = [], [], []
    with torch.inference_mode():
        for images, labels, sequence_ids, _ in tqdm(loader, desc='Validating', leave=False):
            logits = model(images.to(device))
            labels_out.append(labels.numpy())
            probabilities_out.append(torch.sigmoid(logits).cpu().numpy())
            sequence_ids_out.extend(sequence_ids)
    return np.concatenate(labels_out), np.concatenate(probabilities_out), sequence_ids_out

best_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
best_model_path = MODEL_DIR / f'{MODEL_NAME}_best.pt'

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    # Keep frozen BatchNorm layers fixed; only layer4 receives train-mode statistics.
    model.backbone.eval()
    model.backbone.layer4.train()
    model.classifier.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0

    for step, (images, labels, _, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{MAX_EPOCHS}')):
        images = images.to(device)
        labels = labels.float().to(device)
        raw_loss = criterion(model(images), labels)
        (raw_loss / GRADIENT_ACCUMULATION_STEPS).backward()
        loss_sum += raw_loss.item() * len(labels)

        is_last_step = (step + 1) == len(train_loader)
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or is_last_step:
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    validation_labels, validation_probabilities, _ = evaluate(model, validation_loader)
    metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
    record = {
        'epoch': epoch,
        'train_loss': loss_sum / len(train_loader.dataset),
        'validation_accuracy_at_0_5': metrics_at_05['accuracy'],
        'validation_f1_at_0_5': metrics_at_05['f1'],
        'validation_recall_at_0_5': metrics_at_05['recall'],
        'validation_pr_auc': metrics_at_05['pr_auc'],
        'validation_roc_auc': metrics_at_05['roc_auc'],
    }
    history.append(record)
    print(record)

    if record['validation_pr_auc'] > best_pr_auc:
        best_pr_auc = record['validation_pr_auc']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(), 'epoch': epoch,
            'validation_pr_auc': best_pr_auc, 'model_name': MODEL_NAME,
            'base_model': 'A2 ResNet18 + mean-max pooling',
            'fine_tuned_layers': ['backbone.layer4', 'classifier'],
            'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR,
            'num_frames': NUM_FRAMES,
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch: {best_epoch}')
            break

history_path = MODEL_DIR / f'{MODEL_NAME}_training_history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
print(f'Best epoch by validation PR-AUC: {best_epoch}, PR-AUC={best_pr_auc:.4f}')

Epoch 1/5: 100%|██████████| 480/480 [15:51<00:00,  1.98s/it]


{'epoch': 1, 'train_loss': 0.7650121806267028, 'validation_accuracy_at_0_5': 0.7, 'validation_f1_at_0_5': 0.6538461538461539, 'validation_recall_at_0_5': 0.5666666666666667, 'validation_pr_auc': 0.709797398585079, 'validation_roc_auc': 0.7530555555555555}


Epoch 2/5: 100%|██████████| 480/480 [14:29<00:00,  1.81s/it]


{'epoch': 2, 'train_loss': 0.6756994595518335, 'validation_accuracy_at_0_5': 0.7166666666666667, 'validation_f1_at_0_5': 0.6964285714285714, 'validation_recall_at_0_5': 0.65, 'validation_pr_auc': 0.725605054709735, 'validation_roc_auc': 0.7586111111111111}


Epoch 3/5: 100%|██████████| 480/480 [13:59<00:00,  1.75s/it]


{'epoch': 3, 'train_loss': 0.6181303505320102, 'validation_accuracy_at_0_5': 0.675, 'validation_f1_at_0_5': 0.688, 'validation_recall_at_0_5': 0.7166666666666667, 'validation_pr_auc': 0.7474650411633923, 'validation_roc_auc': 0.7530555555555556}


Epoch 4/5: 100%|██████████| 480/480 [11:16<00:00,  1.41s/it]


{'epoch': 4, 'train_loss': 0.6093303796990465, 'validation_accuracy_at_0_5': 0.65, 'validation_f1_at_0_5': 0.5531914893617021, 'validation_recall_at_0_5': 0.43333333333333335, 'validation_pr_auc': 0.7319752839120977, 'validation_roc_auc': 0.7566666666666666}


Epoch 5/5: 100%|██████████| 480/480 [11:08<00:00,  1.39s/it]
                                                             

{'epoch': 5, 'train_loss': 0.5199728682093944, 'validation_accuracy_at_0_5': 0.7083333333333334, 'validation_f1_at_0_5': 0.6666666666666666, 'validation_recall_at_0_5': 0.5833333333333334, 'validation_pr_auc': 0.7461930813473818, 'validation_roc_auc': 0.7558333333333332}
Early stopping at epoch 5; best epoch: 3
Best epoch by validation PR-AUC: 3, PR-AUC=0.7475


In [6]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
validation_labels, validation_probabilities, validation_sequence_ids = evaluate(model, validation_loader)

threshold_table = pd.DataFrame([
    binary_metrics(validation_labels, validation_probabilities, float(threshold))
    for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2)
])
selected_row = threshold_table.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
selected_threshold = float(selected_row['threshold'])
metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
metrics_at_selected_threshold = binary_metrics(validation_labels, validation_probabilities, selected_threshold)

prediction_table = validation_table.set_index('sequence_id').loc[validation_sequence_ids].reset_index()
prediction_table['positive_probability'] = validation_probabilities
prediction_table['prediction_at_0_5'] = (validation_probabilities >= 0.5).astype(int)
prediction_table['prediction_at_selected_threshold'] = (validation_probabilities >= selected_threshold).astype(int)
prediction_table['selected_threshold'] = selected_threshold

predictions_path = MODEL_DIR / f'{MODEL_NAME}_validation_predictions.csv'
threshold_path = MODEL_DIR / f'{MODEL_NAME}_threshold_curve.csv'
metrics_path = MODEL_DIR / f'{MODEL_NAME}_metrics.json'
prediction_table.to_csv(predictions_path, index=False)
threshold_table.to_csv(threshold_path, index=False)

metrics_payload = {
    'model_name': MODEL_NAME,
    'evaluation_scope': 'clip-level validation on fixed V2-W2 sequences; not full-MP4 sliding-window inference',
    'selection_metric': 'validation PR-AUC at the saved checkpoint',
    'best_epoch': int(checkpoint['epoch']),
    'selected_threshold_by_validation_f1': selected_threshold,
    'metrics_at_threshold_0_5': metrics_at_05,
    'metrics_at_selected_threshold': metrics_at_selected_threshold,
    'encoder': 'ResNet18 ImageNet initialized; layer4 fine-tuned',
    'temporal_aggregator': 'mean-max pooling over 16 frame features',
    'input_shape_per_sequence': [NUM_FRAMES, 3, TARGET_HEIGHT, TARGET_WIDTH],
    'train_augmentation': 'sequence-consistent mild crop, horizontal flip, brightness, contrast',
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= 0.5).astype(int), ax=axes[0], colorbar=False)
axes[0].set_title('Validation, threshold = 0.50')
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= selected_threshold).astype(int), ax=axes[1], colorbar=False)
axes[1].set_title(f'Validation, threshold = {selected_threshold:.2f}')
figure.tight_layout()
confusion_path = MODEL_DIR / f'{MODEL_NAME}_confusion_matrices.png'
figure.savefig(confusion_path, dpi=160)
plt.close(figure)

comparison_rows = []
if A2_METRICS_PATH.exists():
    a2_payload = json.loads(A2_METRICS_PATH.read_text(encoding='utf-8'))
    a2_metrics = a2_payload['metrics_at_selected_threshold']
    comparison_rows.append({
        'model': 'A2 ResNet18 frozen + mean-max pooling',
        'f1': a2_metrics['f1'], 'recall': a2_metrics['recall'],
        'precision': a2_metrics['precision'], 'pr_auc': a2_metrics['pr_auc'],
        'threshold': a2_payload['selected_threshold_by_validation_f1'],
    })
comparison_rows.append({
    'model': 'B ResNet18 layer4 fine-tuned + mean-max pooling',
    'f1': metrics_at_selected_threshold['f1'], 'recall': metrics_at_selected_threshold['recall'],
    'precision': metrics_at_selected_threshold['precision'], 'pr_auc': metrics_at_selected_threshold['pr_auc'],
    'threshold': selected_threshold,
})
comparison_table = pd.DataFrame(comparison_rows)
comparison_path = MODEL_DIR / 'v2_a2_vs_finetuned_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)

print('Metrics at threshold 0.50:')
print(metrics_at_05)
print('Metrics at validation-selected threshold:')
print(metrics_at_selected_threshold)
display(comparison_table)
print(f'Model: {best_model_path}')
print(f'Comparison: {comparison_path}')

Metrics at threshold 0.50:
{'threshold': 0.5, 'accuracy': 0.675, 'precision': 0.6615384615384615, 'recall': 0.7166666666666667, 'f1': 0.688, 'roc_auc': 0.7530555555555556, 'pr_auc': 0.7474650411633923, 'confusion_matrix': [[38, 22], [17, 43]]}
Metrics at validation-selected threshold:
{'threshold': 0.2, 'accuracy': 0.6333333333333333, 'precision': 0.5769230769230769, 'recall': 1.0, 'f1': 0.7317073170731707, 'roc_auc': 0.7530555555555556, 'pr_auc': 0.7474650411633923, 'confusion_matrix': [[16, 44], [0, 60]]}


,model,f1,recall,precision,pr_auc,threshold
0,A2 ResNet18 frozen + mean-max pooling,0.787402,0.833333,0.746269,0.742437,0.44
1,B ResNet18 layer4 fine-tuned + mean-max pooling,0.731707,1.000000,0.576923,0.747465,0.20


Model: P:\NexarCollisionData\models_v2\resnet18_meanmax_finetuned_layer4_best.pt
Comparison: P:\NexarCollisionData\models_v2\v2_a2_vs_finetuned_comparison.csv


## تصمیم پس از fine-tuning

B فقط در صورت بهبود F1 نسبت به A2 و حفظ Recall انتخاب می‌شود. اگر نه، A2 را نگه می‌داریم. پس از انتخاب مدل، مرحلهٔ ضروری باقی‌مانده evaluation با sliding-window روی MP4 کامل، threshold final، تحلیل خطا و سپس cross-validation مدل‌های برتر است.